# Free Phases 0–3: recorded batch-eight run
Use only a free GPU and keep this notebook private. This notebook restores the exact original source plus the documented inference-batching amendment. It retains 12,908 training examples and all 2,304 validation questions. No final-test access or RL is included. Run on a fresh runtime, not alongside the active notebook. Source, results, complete trainer checkpoints and restore receipts are backed up privately through the owner's existing Drive authorization. The hosted baseline still needs an owner-confirmed Free tier Gemini project with billing disabled. No completed result is claimed by this template.


In [ ]:
import json, time
SESSION_STARTED=time.monotonic()
import os, pathlib, shutil, zipfile, io, base64, hashlib, subprocess, sys
PROJECT = pathlib.Path('/content/sqlrl_temp/sql-execution-grpo')
PERSIST = pathlib.Path('/content/sqlrl_free_run')
# Upload/extract a prior private output and enter its sqlrl_free_run path.
RESUME_FROM = ''
RUN_LOCAL = True
RUN_GEMINI = False
BILLING_DISABLED_CONFIRMED = False
MAX_REQUESTS = 20
if RESUME_FROM and not PERSIST.exists():
    shutil.copytree(RESUME_FROM, PERSIST)
PERSIST.mkdir(parents=True, exist_ok=True)
PROJECT.mkdir(parents=True, exist_ok=True)
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == 'bf30e56913aabd272ee8c8d759c038e62246e6a66105602e98645a73b099b48e'
snapshot = PERSIST / 'source_snapshot.zip'
if snapshot.exists(): assert snapshot.read_bytes() == payload, 'Resume requires the same source snapshot'
else: snapshot.write_bytes(payload)
# Only results and processed records are included in the output archive.
for relative, target in [('results', PERSIST/'results'), ('data/processed', PERSIST/'processed')]:
    target.mkdir(exist_ok=True)
    link = PROJECT / relative
    if not link.exists():
        link.parent.mkdir(parents=True, exist_ok=True)
        link.symlink_to(target, target_is_directory=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        expected=archive.read(name)
        if name=='src/sqlrl/train_sft/run.py':
            expected=expected.decode().replace('save_strategy="steps", save_steps=100, save_total_limit=2,','save_strategy="steps", save_steps=config.get("checkpoint_steps",100), save_total_limit=2,').encode()
            assert hashlib.sha256(expected).hexdigest()=='9c9fab2b38c75dc85ed3d4acbebb06125cebd3875353ecd50b4e0c8586ef84f7'
        if name=='configs/free/sft.json':
            value=json.loads(expected)
            value['checkpoint_steps']=10
            expected=(json.dumps(value,indent=2)+chr(10)).encode()
        path=PROJECT/name
        if path.exists(): assert path.read_bytes()==expected, 'Different existing source: '+name
        else:
            path.parent.mkdir(parents=True,exist_ok=True)
            path.write_bytes(expected)
os.chdir(PROJECT)
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['SQLRL_FREE_GPU'] = '1'  # Use only on your confirmed free allocation.
os.environ['HF_HOME'] = '/content/sqlrl_temp/sqlrl_hf_cache'
print('Source SHA256:', hashlib.sha256(payload).hexdigest())

ACTIVE_ROOT='results/free_runs_batch8_v2'
OPERATIONAL_BUNDLE=json.loads('{"scripts/run_batched_generation.py": "\\"\\"\\"Frozen-data Qwen generation in recorded, fixed-size inference batches.\\"\\"\\"\\nimport argparse\\nimport json\\nimport shutil\\nimport time\\nfrom pathlib import Path\\n\\nfrom sqlrl.common import seed_everything, sha256, start_run, resume_run, write_json, write_jsonl\\nfrom sqlrl.data.guard import load_records\\nfrom sqlrl.eval.generate import select_examples\\nfrom sqlrl.eval.prompts import messages\\n\\n\\ndef batch_size_from_config(config):\\n    value = config.get(\\"generation_batch_size\\", 8)\\n    if not isinstance(value, int) or isinstance(value, bool) or not 1 <= value <= 16:\\n        raise ValueError(\\"Inference batch size must be an integer between 1 and 16\\")\\n    return value\\n\\n\\ndef validate_prefix(outputs, rows):\\n    if [r[\\"id\\"] for r in outputs] != [r[\\"id\\"] for r in rows[:len(outputs)]]:\\n        raise ValueError(\\"Saved predictions do not match the exact validation prefix\\")\\n    return outputs\\n\\n\\ndef trim_eos(tokens, eos_ids):\\n    end = next((index+1 for index, token in enumerate(tokens) if token in eos_ids), len(tokens))\\n    return tokens[:end]\\n\\n\\ndef generate(config, directory, *, resume=False):\\n    if config.get(\\"backend\\") != \\"transformers\\" or not shutil.which(\\"nvidia-smi\\"):\\n        raise RuntimeError(\\"This batched profile requires a confirmed local NVIDIA GPU\\")\\n    import torch\\n    from transformers import AutoModelForCausalLM, AutoTokenizer\\n    from sqlrl.model_loading import model_kwargs\\n    if not torch.cuda.is_available() or not config.get(\\"revision\\"):\\n        raise RuntimeError(\\"A CUDA GPU and pinned model revision are required\\")\\n    batch_size = batch_size_from_config(config)\\n    rows = load_records(config[\\"data\\"], expected_split=\\"val\\")\\n    if len(rows) != 2304 or any(not r.get(\\"schema\\") or not Path(r[\\"db_path\\"]).is_file() for r in rows):\\n        raise RuntimeError(\\"The full frozen validation set, schemas and databases are required\\")\\n    examples = []\\n    if config.get(\\"few_shot\\", 0):\\n        train = load_records(config[\\"train_data\\"], expected_split=\\"train\\")\\n        if {(r[\\"dataset\\"],r[\\"db_id\\"]) for r in rows} & {(r[\\"dataset\\"],r[\\"db_id\\"]) for r in train}:\\n            raise ValueError(\\"Few-shot training databases overlap validation\\")\\n        examples = select_examples(train,config[\\"few_shot\\"],config[\\"seed\\"])\\n    write_json(directory/\\"few_shot_ids.json\\",[r[\\"id\\"] for r in examples])\\n    outputs = []\\n    if resume and (directory/\\"predictions.jsonl\\").exists():\\n        outputs = [json.loads(line) for line in (directory/\\"predictions.jsonl\\").read_text().splitlines() if line]\\n        validate_prefix(outputs,rows)\\n    tokenizer = AutoTokenizer.from_pretrained(config[\\"model\\"],revision=config[\\"revision\\"])\\n    tokenizer.padding_side = \\"left\\"\\n    tokenizer.pad_token = tokenizer.eos_token\\n    model = AutoModelForCausalLM.from_pretrained(config[\\"model\\"],revision=config[\\"revision\\"],**model_kwargs(config))\\n    if config.get(\\"adapter\\"):\\n        from peft import PeftModel\\n        model = PeftModel.from_pretrained(model,config[\\"adapter\\"])\\n    model.eval()\\n    eos = model.generation_config.eos_token_id\\n    eos_ids = set(eos if isinstance(eos,list) else [eos])\\n    for offset in range(len(outputs),len(rows),batch_size):\\n        batch = rows[offset:offset+batch_size]\\n        began = time.monotonic()\\n        pending, valid = [], []\\n        for row in batch:\\n            prompt = messages(row,examples)\\n            item = {\\"id\\":row[\\"id\\"],\\"messages\\":prompt,\\"answer\\":\\"\\",\\"input_tokens\\":None,\\"output_tokens\\":None,\\n                    \\"cost_usd\\":None,\\"latency_s\\":None,\\"batch_index\\":offset//batch_size,\\"requested_batch_size\\":batch_size}\\n            try:\\n                encoded = tokenizer.apply_chat_template(prompt,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors=\\"pt\\")\\n                ids = encoded[\\"input_ids\\"][0].tolist()\\n                if len(ids) > config[\\"max_prompt_tokens\\"]:\\n                    raise ValueError(\\"Prompt exceeds preregistered context budget\\")\\n                valid.append((len(pending),ids))\\n                item[\\"input_tokens\\"] = len(ids)\\n            except Exception as error:\\n                item[\\"error\\"] = type(error).__name__+\\": \\"+str(error)[:300]\\n            pending.append(item)\\n        if valid:\\n            batch_ids = [ids for _,ids in valid]\\n            inputs = tokenizer.pad({\\"input_ids\\":batch_ids,\\"attention_mask\\":[[1]*len(ids) for ids in batch_ids]},padding=True,return_tensors=\\"pt\\").to(model.device)\\n            torch.cuda.synchronize()\\n            # A CUDA failure stops this batch, preserving the last atomic prefix.\\n            with torch.inference_mode():\\n                generated = model.generate(**inputs,max_new_tokens=config[\\"max_new_tokens\\"],do_sample=False,pad_token_id=tokenizer.eos_token_id)\\n            torch.cuda.synchronize()\\n            for index,(position,_) in enumerate(valid):\\n                tokens = trim_eos(generated[index,inputs[\\"input_ids\\"].shape[-1]:].tolist(),eos_ids)\\n                pending[position].update(answer=tokenizer.decode(tokens,skip_special_tokens=True),output_tokens=len(tokens))\\n        seconds = time.monotonic()-began\\n        for item in pending:\\n            item.update(latency_s=seconds,actual_batch_size=len(valid),amortized_batch_seconds=seconds/len(pending))\\n        outputs.extend(pending)\\n        write_jsonl(directory/\\"predictions.jsonl\\",outputs)\\n        print(json.dumps({\\"saved_predictions\\":len(outputs),\\"total\\":len(rows),\\"batch_seconds\\":round(seconds,3),\\n                          \\"generation_errors\\":sum(\\"error\\" in r for r in outputs)}),flush=True)\\n    write_json(directory/\\"status.json\\",{\\"status\\":\\"completed\\",\\"queries\\":len(outputs),\\"api_budget_debited_usd\\":0,\\n        \\"generation_failures\\":sum(\\"error\\" in r for r in outputs),\\"generation_batch_size\\":batch_size,\\n        \\"profile_id\\":config[\\"profile_id\\"],\\"latency_definition\\":\\"Each request records its shared batch wall time; amortized throughput is a separate field\\"})\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--config\\",required=True)\\n    parser.add_argument(\\"--resume\\",action=\\"store_true\\")\\n    args = parser.parse_args()\\n    config = json.loads(Path(args.config).read_text())\\n    batch_size_from_config(config)\\n    config[\\"data_sha256\\"] = sha256(config[\\"data\\"])\\n    if config.get(\\"train_data\\"):\\n        config[\\"train_data_sha256\\"] = sha256(config[\\"train_data\\"])\\n    config[\\"generation_script_sha256\\"] = sha256(__file__)\\n    directory = resume_run(config[\\"output\\"],config) if args.resume else start_run(config[\\"output\\"],config)\\n    seed_everything(config[\\"seed\\"])\\n    try:\\n        generate(config,directory,resume=args.resume)\\n    except Exception as error:\\n        write_json(directory/\\"status.json\\",{\\"status\\":\\"blocked_or_failed\\",\\"error_type\\":type(error).__name__,\\"reason\\":str(error)[:500]})\\n        raise SystemExit(str(error))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/run_free_batched.py": "\\"\\"\\"Run the recorded batch-eight profile while preserving the serial reference.\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\nfrom pathlib import Path\\n\\nfrom sqlrl.common import sha256, write_json\\nfrom sqlrl.complete_phase03 import data_blockers, report\\n\\n\\ndef main():\\n    parser=argparse.ArgumentParser()\\n    parser.add_argument(\\"--stage\\",choices=[\\"local\\",\\"gemini\\",\\"report\\"],required=True)\\n    parser.add_argument(\\"--root\\",default=\\"results/free_runs_batch8_v2\\")\\n    parser.add_argument(\\"--max-seconds\\",type=int,default=30000)\\n    parser.add_argument(\\"--max-requests\\",type=int,default=20)\\n    args=parser.parse_args()\\n    if args.max_seconds <= 0:\\n        raise ValueError(\\"Session duration must be positive\\")\\n    root=Path(args.root);root.mkdir(parents=True,exist_ok=True)\\n    configs={}\\n    for name in [\\"base_zero\\",\\"base_few\\",\\"sft\\",\\"sft_eval\\",\\"gemini\\"]:\\n        config=json.loads(Path(\\"configs/free_batch8_v2\\",name+\\".json\\").read_text())\\n        config[\\"output\\"]=str(root/(\\"frontier\\" if name==\\"gemini\\" else name))\\n        if name==\\"sft_eval\\":config[\\"adapter\\"]=str(root/\\"sft\\"/\\"adapter\\")\\n        path=root/\\"configs\\"/(name+\\".json\\")\\n        if path.exists() and json.loads(path.read_text()) != config:\\n            raise ValueError(\\"Existing free-run configuration differs; use a new root\\")\\n        write_json(path,config);configs[name]=config\\n    blockers,_=data_blockers(configs[\\"base_zero\\"])\\n    if blockers:\\n        raise RuntimeError(\\"; \\".join(blockers))\\n    deadline=time.monotonic()+args.max_seconds\\n\\n    def run(name, command):\\n        seconds=deadline-time.monotonic()\\n        if seconds <= 0:raise TimeoutError(\\"Notebook session budget exhausted\\")\\n        begin=time.monotonic()\\n        with (root/(name+\\".log\\")).open(\\"a\\") as log:\\n            process=subprocess.Popen([sys.executable,*command],stdout=log,stderr=subprocess.STDOUT,start_new_session=True)\\n            try:\\n                code=process.wait(timeout=seconds)\\n            except subprocess.TimeoutExpired:\\n                os.killpg(process.pid,signal.SIGTERM)\\n                try:process.wait(timeout=10)\\n                except subprocess.TimeoutExpired:\\n                    os.killpg(process.pid,signal.SIGKILL);process.wait()\\n                write_json(root/\\"session_status.json\\",{\\"status\\":\\"paused\\",\\"reason\\":\\"session_time_limit\\",\\"stage\\":name})\\n                raise TimeoutError(\\"Saved stages/checkpoints preserved for the next notebook session\\")\\n        with (root/\\"sessions.jsonl\\").open(\\"a\\") as log:\\n            log.write(json.dumps({\\"stage\\":name,\\"elapsed_s\\":time.monotonic()-begin,\\"returncode\\":code,\\"cash_cost_usd\\":0,\\"cost_basis\\":\\"user-selected free allocation\\"})+\\"\\\\n\\")\\n        if code:raise RuntimeError(f\\"{name} failed; inspect its log. Completed stages remain saved\\")\\n\\n    def evaluate(name):\\n        output=root/(name+\\"_evaluation\\")\\n        predictions=root/name/\\"predictions.jsonl\\"\\n        if (output/\\"summary.json\\").exists():\\n            saved=json.loads((output/\\"config.json\\").read_text())\\n            if saved[\\"predictions_sha256\\"] != sha256(predictions) or saved[\\"data_sha256\\"] != sha256(\\"data/processed/val.jsonl\\"):\\n                raise ValueError(\\"Saved evaluation inputs changed\\")\\n            return\\n        if output.exists():raise RuntimeError(\\"Interrupted evaluation preserved; use a new run root to recompute\\")\\n        run(name+\\"_evaluation\\",[\\"-m\\",\\"sqlrl.eval.run\\",\\"--predictions\\",str(predictions),\\"--output\\",str(output)])\\n\\n    if args.stage==\\"local\\":\\n        if os.getenv(\\"SQLRL_FREE_GPU\\") != \\"1\\":\\n            raise RuntimeError(\\"Confirm a free GPU allocation and set SQLRL_FREE_GPU=1\\")\\n        for name in [\\"base_zero\\",\\"base_few\\",\\"sft\\",\\"sft_eval\\"]:\\n            directory=Path(configs[name][\\"output\\"])\\n            state=json.loads((directory/\\"status.json\\").read_text()) if (directory/\\"status.json\\").exists() else {}\\n            done=state.get(\\"status\\") in {\\"trained\\",\\"completed\\"}\\n            if not done:\\n                if name==\\"sft\\" and not directory.exists() and deadline-time.monotonic()<1800:\\n                    write_json(root/\\"session_status.json\\",{\\"status\\":\\"paused\\",\\"stage\\":\\"sft\\",\\"reason\\":\\"less_than_30_minutes_before_session_limit\\"})\\n                    raise TimeoutError(\\"Completed baselines preserved; start SFT in the next free session with enough checkpoint time\\")\\n                command=([\\"-m\\",\\"sqlrl.train_sft.run\\"] if name==\\"sft\\" else [\\"scripts/run_batched_generation.py\\"])+[\\"--config\\",str(root/\\"configs\\"/(name+\\".json\\"))]\\n                if directory.exists():\\n                    if name==\\"sft\\":\\n                        checkpoints=list((directory/\\"checkpoints\\").glob(\\"checkpoint-*\\"))\\n                        if not checkpoints:raise RuntimeError(\\"No complete SFT checkpoint exists; preserve this attempt and select a new root\\")\\n                        checkpoint=max(checkpoints,key=lambda p:int(p.name.split(\\"-\\")[-1]))\\n                        command += [\\"--resume-from-checkpoint\\",str(checkpoint)]\\n                    else:command += [\\"--resume\\"]\\n                run(name,command)\\n            if name!=\\"sft\\":evaluate(name)\\n    elif args.stage==\\"gemini\\":\\n        directory=Path(configs[\\"gemini\\"][\\"output\\"])\\n        command=[\\"-m\\",\\"sqlrl.eval.free_gemini\\",\\"--config\\",str(root/\\"configs/gemini.json\\"),\\"--max-requests\\",str(args.max_requests)]\\n        if directory.exists():command += [\\"--resume\\"]\\n        run(\\"gemini\\",command)\\n        state=json.loads((directory/\\"status.json\\").read_text())\\n        if state[\\"status\\"]==\\"completed\\":evaluate(\\"frontier\\")\\n    else:\\n        report(root,{\\"base_zero\\":0.0,\\"base_few\\":0.0,\\"sft\\":0.0,\\"sft_eval\\":0.0})\\n    print(\\"Free stage finished; check results before continuing to the next stage.\\")\\n\\n\\nif __name__==\\"__main__\\":\\n    try:main()\\n    except TimeoutError as error:print(str(error))", "scripts/probe_batch_generation.py": "\\"\\"\\"Bounded GPU throughput diagnostic; never a benchmark accuracy result.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport time\\nfrom pathlib import Path\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--output\\", required=True)\\n    parser.add_argument(\\"--count\\", type=int, default=8)\\n    parser.add_argument(\\"--reference\\", default=\\"results/free_runs/base_zero/predictions.jsonl\\")\\n    args = parser.parse_args()\\n    if not 1 <= args.count <= 16:\\n        raise ValueError(\\"Diagnostic count must be between 1 and 16\\")\\n    output = Path(args.output)\\n    if output.exists():\\n        raise ValueError(\\"Preserve the prior diagnostic; choose a new output\\")\\n    from sqlrl.common import seed_everything, write_json\\n    from sqlrl.data.guard import load_records\\n    from sqlrl.eval.prompts import messages\\n    from sqlrl.model_loading import model_kwargs\\n    import torch\\n    from transformers import AutoModelForCausalLM, AutoTokenizer\\n    config = json.loads(Path(\\"configs/free/base_zero.json\\").read_text())\\n    seed_everything(config[\\"seed\\"])\\n    rows = load_records(config[\\"data\\"], expected_split=\\"val\\")[:args.count]\\n    reference = {r[\\"id\\"]:r for r in map(json.loads,Path(args.reference).read_text().splitlines())}\\n    tokenizer = AutoTokenizer.from_pretrained(config[\\"model\\"],revision=config[\\"revision\\"])\\n    tokenizer.padding_side = \\"left\\"\\n    tokenizer.pad_token = tokenizer.eos_token\\n    ids = [tokenizer.apply_chat_template(messages(r),add_generation_prompt=True,tokenize=True,return_dict=True)[\\"input_ids\\"] for r in rows]\\n    if any(len(x) > config[\\"max_prompt_tokens\\"] for x in ids):\\n        raise ValueError(\\"Diagnostic prompt exceeds original context budget\\")\\n    model = AutoModelForCausalLM.from_pretrained(config[\\"model\\"],revision=config[\\"revision\\"],**model_kwargs(config))\\n    model.eval()\\n    eos = model.generation_config.eos_token_id\\n    eos = set(eos if isinstance(eos,list) else [eos])\\n    receipt = {\\"diagnostic_only\\":True,\\"phase3_complete\\":False,\\"model\\":config[\\"model\\"],\\"revision\\":config[\\"revision\\"],\\"ids\\":[r[\\"id\\"] for r in rows],\\"runs\\":[]}\\n    for batch_size in [4,8]:\\n        torch.cuda.reset_peak_memory_stats()\\n        outputs = []\\n        total_seconds = 0.0\\n        for start in range(0,len(rows),batch_size):\\n            current = rows[start:start+batch_size]\\n            batch_ids = ids[start:start+batch_size]\\n            inputs = tokenizer.pad({\\"input_ids\\":batch_ids,\\"attention_mask\\":[[1]*len(x) for x in batch_ids]},padding=True,return_tensors=\\"pt\\")\\n            for index, expected in enumerate(ids[start:start+batch_size]):\\n                actual = inputs[\\"input_ids\\"][index][inputs[\\"attention_mask\\"][index].bool()].tolist()\\n                if actual != expected:\\n                    raise ValueError(\\"Batched prompt tokens differ from original inputs\\")\\n            inputs = inputs.to(model.device)\\n            torch.cuda.synchronize()\\n            began = time.monotonic()\\n            with torch.inference_mode():\\n                generated = model.generate(**inputs,max_new_tokens=config[\\"max_new_tokens\\"],do_sample=False,pad_token_id=tokenizer.eos_token_id)\\n            torch.cuda.synchronize()\\n            seconds = time.monotonic()-began\\n            total_seconds += seconds\\n            for index,row in enumerate(current):\\n                tokens = generated[index,inputs[\\"input_ids\\"].shape[-1]:].tolist()\\n                end = next((i+1 for i,t in enumerate(tokens) if t in eos),len(tokens))\\n                tokens = tokens[:end]\\n                answer = tokenizer.decode(tokens,skip_special_tokens=True)\\n                prior = reference.get(row[\\"id\\"])\\n                outputs.append({\\"id\\":row[\\"id\\"],\\"input_tokens\\":len(ids[start+index]),\\"output_tokens\\":len(tokens),\\"answer_sha256\\":hashlib.sha256(answer.encode()).hexdigest(),\\"matches_reference\\":answer==prior[\\"answer\\"] if prior and \\"error\\" not in prior else None})\\n        prior_seconds = sum(reference[r[\\"id\\"]][\\"latency_s\\"] for r in rows if r[\\"id\\"] in reference)\\n        receipt[\\"runs\\"].append({\\"batch_size\\":batch_size,\\"questions\\":len(outputs),\\"generation_seconds\\":total_seconds,\\"amortized_seconds_per_question\\":total_seconds/len(outputs),\\"reference_seconds_for_same_questions\\":prior_seconds,\\"speedup_vs_saved_serial_latencies\\":prior_seconds/total_seconds,\\"exact_answer_matches\\":sum(r[\\"matches_reference\\"] is True for r in outputs),\\"peak_allocated_bytes\\":torch.cuda.max_memory_allocated(),\\"peak_reserved_bytes\\":torch.cuda.max_memory_reserved(),\\"outputs\\":outputs})\\n        write_json(output,receipt)\\n        print(json.dumps(receipt[\\"runs\\"][-1]),flush=True)\\n    del model\\n    torch.cuda.empty_cache()\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "scripts/colab_private_backup.py": "# Colab-only private backup helper. Uses the owner\'s existing Drive authorization.\\nimport hashlib, json, pathlib, threading, time, zipfile\\nfrom googleapiclient.discovery import build\\nfrom googleapiclient.http import MediaFileUpload\\n\\ndef save_private_checkpoint():\\n    api=build(\'drive\',\'v3\',credentials=drive_credentials,cache_discovery=False)\\n    folder_name=\'SQLRL-free-phase03-9c9fab2b\'\\n    folders=api.files().list(q=\\"name=\'\\"+folder_name+\\"\' and mimeType=\'application/vnd.google-apps.folder\' and trashed=false and \'root\' in parents\\",fields=\'files(id,name)\').execute()[\'files\']\\n    assert len(folders)<=1,\'Ambiguous project folders; preserve files for review\'\\n    folder_id=folders[0][\'id\'] if folders else api.files().create(body={\'name\':folder_name,\'mimeType\':\'application/vnd.google-apps.folder\'},fields=\'id\').execute()[\'id\']\\n    checkpoint_roots=[PERSIST/\'results\'/profile/\'sft/checkpoints\' for profile in [\'free_runs\',\'free_runs_batch8_v2\']]\\n    latest={}\\n    for root in checkpoint_roots:\\n        complete=[]\\n        for directory in root.glob(\'checkpoint-*\'):\\n            required=[\'trainer_state.json\',\'optimizer.pt\',\'scheduler.pt\',\'rng_state.pth\',\'adapter_model.safetensors\',\'adapter_config.json\']\\n            if all((directory/name).is_file() for name in required) and max(p.stat().st_mtime for p in directory.rglob(\'*\') if p.is_file())<time.time()-30:\\n                complete.append(directory)\\n        latest[root]=max(complete,key=lambda p:int(p.name.rsplit(\'-\',1)[1])) if complete else None\\n    transfer=pathlib.Path(\'/content/sqlrl_checkpoint_transfer.zip\')\\n    with zipfile.ZipFile(transfer,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as archive:\\n        for path in PERSIST.rglob(\'*\'):\\n            if not path.is_file() or path.is_symlink(): continue\\n            if any(root in path.parents and (selected is None or selected not in path.parents) for root,selected in latest.items()): continue\\n            archive.write(path,path.relative_to(PERSIST))\\n    with zipfile.ZipFile(transfer) as archive: assert archive.testzip() is None\\n    size=transfer.stat().st_size\\n    assert size<=1024**3,\'Pause above the declared 1 GiB transfer cap\'\\n    existing=api.files().list(q=\\"\'\\"+folder_id+\\"\' in parents and name=\'phase03-latest.zip\' and trashed=false\\",fields=\'files(id,size)\').execute()[\'files\']\\n    assert len(existing)<=1,\'Ambiguous checkpoint files; stop for review\'\\n    quota=api.about().get(fields=\'storageQuota\').execute()[\'storageQuota\']\\n    available=int(quota.get(\'limit\',0))-int(quota.get(\'usage\',0))+(int(existing[0][\'size\']) if existing else 0)\\n    assert size<available,\'Insufficient existing Drive storage; never purchase more\'\\n    sha=hashlib.sha256()\\n    md5=hashlib.md5()\\n    with transfer.open(\'rb\') as stream:\\n        for block in iter(lambda:stream.read(8*1024**2),b\'\'):\\n            sha.update(block); md5.update(block)\\n    media=MediaFileUpload(str(transfer),mimetype=\'application/zip\',resumable=True,chunksize=8*1024**2)\\n    metadata={\'name\':\'phase03-latest.zip\',\'appProperties\':{\'sha256\':sha.hexdigest(),\'profile\':\'student_free_nf4_fp16_batch8_v2\'}}\\n    if existing:\\n        request=api.files().update(fileId=existing[0][\'id\'],body=metadata,media_body=media,fields=\'id,size,md5Checksum,webViewLink\')\\n    else:\\n        metadata[\'parents\']=[folder_id]\\n        request=api.files().create(body=metadata,media_body=media,fields=\'id,size,md5Checksum,webViewLink\')\\n    uploaded=None\\n    while uploaded is None:\\n        progress,uploaded=request.next_chunk(num_retries=0)\\n    assert uploaded[\'md5Checksum\']==md5.hexdigest() and int(uploaded[\'size\'])==size\\n    return {\'status\':\'verified_private_drive_backup\',\'sha256\':sha.hexdigest(),\'bytes\':size,\'drive_file\':uploaded,\\n            \'checkpoints\':{str(root.relative_to(PERSIST)):selected.name if selected else None for root,selected in latest.items()},\'phase3_complete\':False}\\n\\ndef backup_loop():\\n    while not backup_stop.is_set():\\n        if \'drive_quota\' in globals():\\n            try: receipt=save_private_checkpoint()\\n            except Exception as error: receipt={\'status\':\'backup_paused\',\'reason\':type(error).__name__+\': \'+str(error)[:300]}\\n            target=PERSIST/\'backup_status.json\'\\n            temporary=target.with_suffix(\'.tmp\')\\n            temporary.write_text(json.dumps(receipt,indent=2)+chr(10))\\n            temporary.replace(target)\\n        backup_stop.wait(300)\\n\\nif \'backup_controller\' in globals() and backup_controller.is_alive():\\n    backup_stop.set()\\n    backup_controller.join(timeout=30)\\n    if backup_controller.is_alive():\\n        backup_stop.clear()\\n        raise RuntimeError(\'Existing backup is still finishing; retry worker replacement later\')\\nbackup_stop=threading.Event()\\nbackup_controller=threading.Thread(target=backup_loop,daemon=True,name=\'private_checkpoint_backup_v2\')\\nbackup_controller.start()\\nprint(\'Private backup worker covers both recorded runs; newest full checkpoint per run only.\',flush=True)\\n", "scripts/colab_restore_private.py": "\\"\\"\\"Restore only the private project archive through the owner\'s Drive service.\\"\\"\\"\\nimport hashlib\\nimport json\\nimport pathlib\\nimport shutil\\nimport stat\\nimport zipfile\\n\\nBASE_SOURCE_SHA256 = \\"bf30e56913aabd272ee8c8d759c038e62246e6a66105602e98645a73b099b48e\\"\\n\\ndef stream_digest(stream, algorithm=\\"sha256\\"):\\n    result=hashlib.new(algorithm)\\n    for block in iter(lambda:stream.read(8*1024**2),b\\"\\"):\\n        result.update(block)\\n    return result.hexdigest()\\n\\ndef validated_members(archive):\\n    members=archive.infolist()\\n    seen=set()\\n    total=0\\n    for member in members:\\n        path=pathlib.PurePosixPath(member.filename)\\n        if path.is_absolute() or \\"..\\" in path.parts or not path.parts or \\"\\\\\\\\\\" in member.filename:\\n            raise ValueError(\\"Unsafe archive member path\\")\\n        if stat.S_ISLNK(member.external_attr >> 16) or member.filename in seen:\\n            raise ValueError(\\"Symlink or duplicate archive member\\")\\n        seen.add(member.filename)\\n        total+=member.file_size\\n    if total > 8*1024**3:\\n        raise ValueError(\\"Archive expands beyond the declared 8 GiB restore cap\\")\\n    return members\\n\\ndef restore_private_checkpoint(persist,service):\\n    from googleapiclient.http import MediaIoBaseDownload\\n    persist=pathlib.Path(persist).resolve()\\n    folders=service.files().list(q=\\"name=\'SQLRL-free-phase03-9c9fab2b\' and mimeType=\'application/vnd.google-apps.folder\' and trashed=false and \'root\' in parents\\",fields=\\"files(id)\\").execute()[\\"files\\"]\\n    if not folders:\\n        return {\\"status\\":\\"no_prior_project_backup\\",\\"phase3_complete\\":False}\\n    if len(folders)!=1:\\n        raise ValueError(\\"Ambiguous project folders; do not overwrite\\")\\n    files=service.files().list(q=\\"\'\\"+folders[0][\\"id\\"]+\\"\' in parents and name=\'phase03-latest.zip\' and trashed=false\\",fields=\\"files(id,size,md5Checksum,webViewLink,appProperties)\\").execute()[\\"files\\"]\\n    if not files:\\n        return {\\"status\\":\\"no_prior_checkpoint_file\\",\\"phase3_complete\\":False}\\n    if len(files)!=1:\\n        raise ValueError(\\"Ambiguous checkpoint files; do not overwrite\\")\\n    remote=files[0]\\n    size=int(remote[\\"size\\"])\\n    if not 0<size<=1024**3:\\n        raise ValueError(\\"Backup exceeds the 1 GiB transfer cap\\")\\n    transfer=persist.parent/\\"sqlrl_restore_transfer.zip\\"\\n    with transfer.open(\\"wb\\") as stream:\\n        download=MediaIoBaseDownload(stream,service.files().get_media(fileId=remote[\\"id\\"]),chunksize=8*1024**2)\\n        done=False\\n        while not done:\\n            unused,done=download.next_chunk(num_retries=0)\\n    with transfer.open(\\"rb\\") as stream: md5=stream_digest(stream,\\"md5\\")\\n    with transfer.open(\\"rb\\") as stream: sha=stream_digest(stream)\\n    if transfer.stat().st_size!=size or md5!=remote[\\"md5Checksum\\"] or remote.get(\\"appProperties\\",{}).get(\\"sha256\\",sha)!=sha:\\n        raise ValueError(\\"Downloaded archive checksum differs from Drive\\")\\n    with zipfile.ZipFile(transfer) as archive:\\n        members=validated_members(archive)\\n        if hashlib.sha256(archive.read(\\"source_snapshot.zip\\")).hexdigest()!=BASE_SOURCE_SHA256:\\n            raise ValueError(\\"Backup belongs to a different frozen source snapshot\\")\\n        if archive.testzip() is not None:\\n            raise ValueError(\\"Backup ZIP integrity check failed\\")\\n        if shutil.disk_usage(persist).free < sum(m.file_size for m in members)+1024**3:\\n            raise ValueError(\\"Insufficient local restore space\\")\\n        for member in members:\\n            target=persist/member.filename\\n            if not target.resolve().is_relative_to(persist) or any(p.is_symlink() for p in [target,*target.parents] if p.is_relative_to(persist)):\\n                raise ValueError(\\"Restore target escapes its project folder\\")\\n            if target.exists() and not member.is_dir():\\n                with target.open(\\"rb\\") as local,archive.open(member) as saved:\\n                    if stream_digest(local)!=stream_digest(saved):\\n                        raise ValueError(\\"Preserve different existing evidence: \\"+member.filename)\\n        for member in members:\\n            target=persist/member.filename\\n            if member.is_dir(): target.mkdir(parents=True,exist_ok=True)\\n            elif not target.exists():\\n                target.parent.mkdir(parents=True,exist_ok=True)\\n                temporary=target.with_name(target.name+\\".restore_tmp\\")\\n                with archive.open(member) as saved,temporary.open(\\"wb\\") as local:\\n                    shutil.copyfileobj(saved,local,length=1024**2)\\n                temporary.replace(target)\\n    receipt={\\"status\\":\\"restored_private_drive_backup\\",\\"bytes\\":size,\\"sha256\\":sha,\\"md5Checksum\\":md5,\\"drive_file\\":remote,\\"phase3_complete\\":False}\\n    target=persist/\\"restore_receipt_current.json\\"\\n    if target.exists():\\n        target=persist/(\\"restore_receipt_\\"+sha[:16]+\\".json\\")\\n    if target.exists() and json.loads(target.read_text())!=receipt:\\n        raise ValueError(\\"Preserve an existing different restore receipt\\")\\n    target.write_text(json.dumps(receipt,indent=2)+\\"\\\\n\\")\\n    return receipt\\n", "tests/test_batched_generation.py": "\\"\\"\\"Resume order and token accounting for the optional batched profile.\\"\\"\\"\\nimport pytest\\nimport importlib.util\\nfrom pathlib import Path\\n_spec=importlib.util.spec_from_file_location(\\"batched_generation_fixture\\",Path(__file__).resolve().parents[1]/\\"scripts/run_batched_generation.py\\")\\n_module=importlib.util.module_from_spec(_spec)\\n_spec.loader.exec_module(_module)\\nbatch_size_from_config,trim_eos,validate_prefix=_module.batch_size_from_config,_module.trim_eos,_module.validate_prefix\\n\\ndef test_multiple_stop_ids_preserve_first_stop_token():\\n    assert trim_eos([8,9,2,4,3,3],{2,3}) == [8,9,2]\\n    assert trim_eos([8,9,3,4],{2,3}) == [8,9,3]\\n\\ndef test_capped_output_without_stop_is_preserved():\\n    tokens = list(range(512))\\n    assert trim_eos(tokens,{999}) == tokens\\n    assert trim_eos([],{999}) == []\\n\\n@pytest.mark.parametrize(\\"saved\\",[\\n    [{\\"id\\":\\"b\\"}],\\n    [{\\"id\\":\\"a\\"},{\\"id\\":\\"a\\"}],\\n    [{\\"id\\":\\"a\\"},{\\"id\\":\\"b\\"},{\\"id\\":\\"extra\\"}],\\n])\\ndef test_resume_rejects_skipped_duplicate_or_extra_rows(saved):\\n    with pytest.raises(ValueError,match=\\"exact validation prefix\\"):\\n        validate_prefix(saved,[{\\"id\\":\\"a\\"},{\\"id\\":\\"b\\"}])\\n\\ndef test_resume_accepts_only_an_ordered_prefix():\\n    rows=[{\\"id\\":\\"a\\"},{\\"id\\":\\"b\\"}]\\n    saved=[{\\"id\\":\\"a\\",\\"answer\\":\\"SELECT 1\\"}]\\n    assert validate_prefix(saved,rows) is saved\\n    assert validate_prefix([],rows) == []\\n\\n@pytest.mark.parametrize(\\"value\\",[0,17,-1,True,2.5,\\"8\\"])\\ndef test_invalid_batch_size_cannot_silently_change_execution(value):\\n    with pytest.raises(ValueError,match=\\"batch size\\"):\\n        batch_size_from_config({\\"generation_batch_size\\":value})\\n\\ndef test_declared_batch_size_is_retained():\\n    assert batch_size_from_config({\\"generation_batch_size\\":8}) == 8\\n\\ndef test_batch_profile_retains_all_frozen_experiment_settings():\\n    import json\\n    from pathlib import Path\\n    for name in [\\"base_zero\\",\\"base_few\\",\\"sft\\",\\"sft_eval\\",\\"gemini\\"]:\\n        serial=json.loads(Path(\\"configs/free\\",name+\\".json\\").read_text())\\n        batched=json.loads(Path(\\"configs/free_batch8_v2\\",name+\\".json\\").read_text())\\n        operational={\\"profile_id\\",\\"output\\",\\"adapter\\",\\"generation_batch_size\\"}\\n        assert {k:v for k,v in serial.items() if k not in operational} == {k:v for k,v in batched.items() if k not in operational}\\n        assert batched[\\"profile_id\\"] == \\"student_free_nf4_fp16_batch8_v2\\"\\n        if name in {\\"base_zero\\",\\"base_few\\",\\"sft_eval\\"}:\\n            assert batched[\\"generation_batch_size\\"] == 8\\n", "tests/test_colab_restore.py": "import io, stat, zipfile\\nimport pytest\\nimport importlib.util\\nfrom pathlib import Path\\n_spec=importlib.util.spec_from_file_location(\\"colab_restore_fixture\\",Path(__file__).resolve().parents[1]/\\"scripts/colab_restore_private.py\\")\\n_module=importlib.util.module_from_spec(_spec)\\n_spec.loader.exec_module(_module)\\nvalidated_members=_module.validated_members\\n\\n@pytest.mark.parametrize(\\"name\\",[\\"../outside\\",\\"/absolute\\",\\"a/../../outside\\",\\"a\\\\\\\\\\\\\\\\escape\\"])\\ndef test_restore_rejects_escaping_paths(name):\\n    data=io.BytesIO()\\n    with zipfile.ZipFile(data,\\"w\\") as archive: archive.writestr(name,b\\"x\\")\\n    with zipfile.ZipFile(data) as archive:\\n        with pytest.raises(ValueError,match=\\"Unsafe\\"):\\n            validated_members(archive)\\n\\ndef test_restore_rejects_symlinks():\\n    data=io.BytesIO()\\n    member=zipfile.ZipInfo(\\"linked\\")\\n    member.create_system=3\\n    member.external_attr=(stat.S_IFLNK|0o777)<<16\\n    with zipfile.ZipFile(data,\\"w\\") as archive: archive.writestr(member,b\\"../outside\\")\\n    with zipfile.ZipFile(data) as archive:\\n        with pytest.raises(ValueError,match=\\"Symlink\\"):\\n            validated_members(archive)\\n\\ndef test_restore_accepts_ordinary_project_members():\\n    data=io.BytesIO()\\n    with zipfile.ZipFile(data,\\"w\\") as archive:\\n        archive.writestr(\\"results/\\",b\\"\\")\\n        archive.writestr(\\"results/free_runs_batch8_v2/base_zero/predictions.jsonl\\",b\'{\\"id\\":\\"a\\"}\\\\n\')\\n    with zipfile.ZipFile(data) as archive:\\n        assert len(validated_members(archive))==2\\n", "docs/BATCHED_FREE_RUN.md": "# Recorded batch-eight free profile\\n\\nThe optional student_free_nf4_fp16_batch8_v2 run changes inference batching only.\\nIt preserves the pinned Qwen revision, NF4/FP16 weights and compute, all 12,908\\neligible training IDs and 2,304 validation IDs, seed, prompts, three-shot\\nselection, context/output limits, SQL verifier and training hyperparameters.\\nOfficial final-test data remains sealed. This is a separately recorded\\noperational amendment to the serial student_free_nf4_fp16_v1 profile.\\n\\n## Measured diagnostic\\n\\nEight fixed first validation questions were tested for token identity and exact\\nanswer equality against saved serial outputs; gold answers were not used to\\nselect a batch size. Batch four took 10.025393 seconds; batch eight took\\n6.903841 seconds, compared with 28.520275 seconds for the same serial questions.\\nAll eight answers matched in both tests. Batch eight used a peak 3,384,800,768\\nbytes of allocated GPU memory in its own process. The earlier serial worker\\nremained allocated but was paused during measurement, then resumed in finally.\\nThe source and receipt identify this as a diagnostic, not an accuracy result.\\nA first tokenization preflight failed before generation and its log was kept.\\n\\nThe observed batch-eight speedup was 4.131 times on these eight questions.\\nThis is not a guarantee for the entire corpus or longer few-shot prompts.\\n\\n## Evidence and resumption\\n\\nUse a new root results/free_runs_batch8_v2; do not combine serial and batched\\npredictions. Preserve the original serial run, source and configuration.\\nThe source under src is unchanged. Batched generation additionally records\\nthe SHA256 of scripts/run_batched_generation.py in each immutable run config,\\nso changing that script prevents resuming a different implementation.\\nPredictions are written atomically after each completed batch and must match\\nthe exact frozen validation prefix on resume. A CUDA error stops the batch\\nand preserves that prefix. It does not fill the rest with manufactured errors.\\n\\nEach prediction\'s latency_s is its shared batch wall time. The separately\\nnamed amortized_batch_seconds field measures throughput. Do not compare these\\nlatencies as if the batched run served isolated serial requests. Input-token\\ncounts exclude padding; output-token counts include the first EOS and exclude\\nsubsequent padding. The original 4,096 input and 512 output limits are retained.\\n\\nThe training command remains sqlrl.train_sft.run with microbatch one,\\naccumulation 16, one epoch, learning rate 0.0002, maximum length 4,608 and\\nthe same LoRA settings. Ten-step checkpoints remain enabled. A fresh SFT stage\\nis deferred if fewer than 30 minutes remain in the bounded session, preserving\\ncompleted baselines and avoiding an attempt with no complete checkpoint.\\n\\nRun scripts/run_free_batched.py --stage local with a session bound within the\\nprovider\'s actual free allowance. Reuse the same source, config, root and full\\ntrainer checkpoint when continuing. The hosted Gemini baseline still uses the\\ndocumented REST profile, free project and owner-managed credential; no paid\\nfallback or undocumented credential route is introduced.\\n", "configs/free_batch8_v2/base_zero.json": "{\\n  \\"model\\": \\"Qwen/Qwen3-4B-Instruct-2507\\",\\n  \\"revision\\": \\"cdbee75f17c01a7cc42f958dc650907174af0554\\",\\n  \\"backend\\": \\"transformers\\",\\n  \\"data\\": \\"data/processed/val.jsonl\\",\\n  \\"train_data\\": \\"data/processed/train_eligible.jsonl\\",\\n  \\"seed\\": 42,\\n  \\"max_prompt_tokens\\": 4096,\\n  \\"max_new_tokens\\": 512,\\n  \\"few_shot\\": 0,\\n  \\"output\\": \\"results/free_runs_batch8_v2/base_zero\\",\\n  \\"gpu_price_usd_per_hour\\": 0,\\n  \\"profile_id\\": \\"student_free_nf4_fp16_batch8_v2\\",\\n  \\"precision\\": \\"fp16\\",\\n  \\"quantization\\": \\"nf4\\",\\n  \\"generation_batch_size\\": 8\\n}\\n", "configs/free_batch8_v2/base_few.json": "{\\n  \\"model\\": \\"Qwen/Qwen3-4B-Instruct-2507\\",\\n  \\"revision\\": \\"cdbee75f17c01a7cc42f958dc650907174af0554\\",\\n  \\"backend\\": \\"transformers\\",\\n  \\"data\\": \\"data/processed/val.jsonl\\",\\n  \\"train_data\\": \\"data/processed/train_eligible.jsonl\\",\\n  \\"seed\\": 42,\\n  \\"max_prompt_tokens\\": 4096,\\n  \\"max_new_tokens\\": 512,\\n  \\"few_shot\\": 3,\\n  \\"output\\": \\"results/free_runs_batch8_v2/base_few\\",\\n  \\"gpu_price_usd_per_hour\\": 0,\\n  \\"profile_id\\": \\"student_free_nf4_fp16_batch8_v2\\",\\n  \\"precision\\": \\"fp16\\",\\n  \\"quantization\\": \\"nf4\\",\\n  \\"generation_batch_size\\": 8\\n}\\n", "configs/free_batch8_v2/sft.json": "{\\n  \\"model\\": \\"Qwen/Qwen3-4B-Instruct-2507\\",\\n  \\"revision\\": \\"cdbee75f17c01a7cc42f958dc650907174af0554\\",\\n  \\"seed\\": 42,\\n  \\"train_data\\": \\"data/processed/train_eligible.jsonl\\",\\n  \\"val_data\\": \\"data/processed/val.jsonl\\",\\n  \\"output\\": \\"results/free_runs_batch8_v2/sft\\",\\n  \\"epochs\\": 1,\\n  \\"learning_rate\\": 0.0002,\\n  \\"gradient_accumulation_steps\\": 16,\\n  \\"max_length\\": 4608,\\n  \\"gpu_price_usd_per_hour\\": 0,\\n  \\"profile_id\\": \\"student_free_nf4_fp16_batch8_v2\\",\\n  \\"precision\\": \\"fp16\\",\\n  \\"quantization\\": \\"nf4\\",\\n  \\"checkpoint_steps\\": 10\\n}\\n", "configs/free_batch8_v2/sft_eval.json": "{\\n  \\"model\\": \\"Qwen/Qwen3-4B-Instruct-2507\\",\\n  \\"revision\\": \\"cdbee75f17c01a7cc42f958dc650907174af0554\\",\\n  \\"backend\\": \\"transformers\\",\\n  \\"data\\": \\"data/processed/val.jsonl\\",\\n  \\"train_data\\": \\"data/processed/train_eligible.jsonl\\",\\n  \\"seed\\": 42,\\n  \\"max_prompt_tokens\\": 4096,\\n  \\"max_new_tokens\\": 512,\\n  \\"few_shot\\": 0,\\n  \\"output\\": \\"results/free_runs_batch8_v2/sft_eval\\",\\n  \\"gpu_price_usd_per_hour\\": 0,\\n  \\"adapter\\": \\"results/free_runs_batch8_v2/sft/adapter\\",\\n  \\"profile_id\\": \\"student_free_nf4_fp16_batch8_v2\\",\\n  \\"precision\\": \\"fp16\\",\\n  \\"quantization\\": \\"nf4\\",\\n  \\"generation_batch_size\\": 8\\n}\\n", "configs/free_batch8_v2/gemini.json": "{\\n  \\"profile_id\\": \\"student_free_nf4_fp16_batch8_v2\\",\\n  \\"model\\": \\"gemini-3.8-flash\\",\\n  \\"endpoint\\": \\"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.8-flash:generateContent\\",\\n  \\"data\\": \\"data/processed/val.jsonl\\",\\n  \\"seed\\": 42,\\n  \\"max_prompt_tokens\\": 4096,\\n  \\"max_new_tokens\\": 512,\\n  \\"min_interval_s\\": 15,\\n  \\"output\\": \\"results/free_runs_batch8_v2/frontier\\",\\n  \\"budget_usd\\": 0,\\n  \\"input_usd_per_million\\": 0,\\n  \\"output_usd_per_million\\": 0,\\n  \\"billing_requirement\\": \\"owner-confirmed Free tier with billing disabled\\",\\n  \\"thinking_level\\": \\"LOW\\",\\n  \\"completion_limit_includes_thinking\\": true\\n}\\n"}')
for name,content in OPERATIONAL_BUNDLE.items():
    target=PROJECT/name
    if target.exists(): assert target.read_text()==content, 'Different operational source: '+name
    else:
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_text(content)
(PERSIST/'resume_bundle_5d6b0b4b7722359e.json').write_text(json.dumps(OPERATIONAL_BUNDLE))


In [ ]:
# Official PyPI and PyTorch CUDA 12.6 wheels; no cloud billing setup.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.13.7'], check=True)
ENV_DIR = '/content/sqlrl_temp/sqlrl_venv'
if not pathlib.Path(ENV_DIR, 'bin/python').exists():
    subprocess.run(['uv', 'venv', '--python', '3.12', ENV_DIR], check=True)
PY = str(pathlib.Path(ENV_DIR, 'bin/python'))
subprocess.run(['uv', 'pip', 'sync', '--python', PY, 'requirements-free.lock', '--extra-index-url', 'https://download.pytorch.org/whl/cu126', '--index-strategy', 'unsafe-best-match'], check=True)
subprocess.run(['uv', 'pip', 'install', '--python', PY, '--no-deps', '-e', '.'], check=True)
subprocess.run([PY, '-m', 'pytest'], check=True)
if RUN_LOCAL:
    subprocess.run([PY, '-c', "import torch; assert torch.cuda.is_available(), 'Select a free GPU in Colab Runtime > Change runtime type'; p=torch.cuda.get_device_properties(0); print(torch.__version__,p.name,p.total_memory,torch.cuda.get_arch_list()); x=torch.ones(8,device='cuda'); print((x+x).sum().item())"], check=True)


In [ ]:
# Public Google APIs only. Credential values remain inside this runtime.
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
drive_credentials,unused_project=google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive_service=build('drive','v3',credentials=drive_credentials,cache_discovery=False)
drive_quota=drive_service.about().get(fields='storageQuota').execute()['storageQuota']
from scripts.colab_restore_private import restore_private_checkpoint
restore_receipt=restore_private_checkpoint(PERSIST,drive_service)
print(json.dumps(restore_receipt,indent=2))


In [ ]:
# Private archive contains only project outputs, processed records and source.
backup_helper_source=(PROJECT/'scripts/colab_private_backup.py').read_text()
(PERSIST/'colab_private_backup.py').write_text(backup_helper_source)
exec(compile(backup_helper_source,'colab_private_backup.py','exec'),globals())


In [ ]:
# TRAIN-only preparation; never run concurrently with the local model controller.
assert not globals().get('model_controller') or not model_controller.is_alive()
assert not globals().get('prep_process') or prep_process.poll() is not None
scratch=pathlib.Path('/content/sqlrl_temp/sqlrl_data')
scratch.mkdir(parents=True,exist_ok=True)
needed_gib=10 if any((scratch/'databases').glob('*/*/*.sqlite')) else 70
assert shutil.disk_usage(scratch).free>needed_gib*1024**3,'Insufficient free scratch space'
for name in ['raw','databases']:
    target=scratch/name
    target.mkdir(exist_ok=True)
    link=PROJECT/'data'/name
    if not link.exists():
        link.parent.mkdir(parents=True,exist_ok=True)
        if link.is_symlink(): link.unlink()
        link.symlink_to(target,target_is_directory=True)
prep_log=PERSIST/'prepare-data.log'
prep_stream=prep_log.open('a')
prep_process=subprocess.Popen([PY,'-u','scripts/prepare_free_data.py'],cwd=PROJECT,stdout=prep_stream,stderr=subprocess.STDOUT,start_new_session=True)
print('TRAIN preparation PID:',prep_process.pid)


In [ ]:
# Bounded local work waits for verified TRAIN data and resumes exact saved prefixes.
import threading
assert not globals().get('model_controller') or not model_controller.is_alive()
def controller_status(value):
    path=PERSIST/'controller_status.json'
    temporary=path.with_suffix('.tmp')
    temporary.write_text(json.dumps(value,indent=2)+chr(10))
    temporary.replace(path)
def continue_batch8_experiments():
    try:
        controller_status({'status':'waiting_for_verified_train_data','run_root':ACTIVE_ROOT,'phase3_complete':False})
        assert prep_process.wait()==0,'TRAIN preparation failed; inspect prepare-data.log'
        seconds=min(10800,int(14400-(time.monotonic()-SESSION_STARTED)))
        assert seconds>=900,'Not enough remaining session time'
        with (PERSIST/'local-batch8-run.log').open('a') as stream:
            process=subprocess.Popen([PY,'-u','scripts/run_free_batched.py','--stage','local','--root',ACTIVE_ROOT,'--max-seconds',str(seconds)],cwd=PROJECT,stdout=stream,stderr=subprocess.STDOUT,start_new_session=True)
            controller_status({'status':'running_local_experiments','run_root':ACTIVE_ROOT,'pid':process.pid,'bounded_seconds':seconds,'cash_budget_usd':0,'phase3_complete':False})
            code=process.wait()
        required={'base_zero':'completed','base_few':'completed','sft':'trained','sft_eval':'completed'}
        states={}
        for name in required:
            path=PROJECT/ACTIVE_ROOT/name/'status.json'
            states[name]=json.loads(path.read_text()).get('status') if path.exists() else 'pending'
        done=all(states[name]==expected for name,expected in required.items())
        controller_status({'status':'local_stages_completed' if done else 'paused_or_failed','run_root':ACTIVE_ROOT,'returncode':code,'stages':states,'hosted_baseline':'pending','phase3_complete':False})
    except Exception as error:
        controller_status({'status':'blocked','run_root':ACTIVE_ROOT,'reason':type(error).__name__+': '+str(error)[:500],'phase3_complete':False})
if RUN_LOCAL:
    model_controller=threading.Thread(target=continue_batch8_experiments,daemon=True,name='free_batch8_model_controller')
    model_controller.start()
print('Inspect the next cell for actual progress.')


In [ ]:
# Re-run this inspection cell as needed; it does not launch model work.
print('TRAIN return code:',prep_process.poll())
print(prep_log.read_text(errors='replace')[-600:])
for filename in ['controller_status.json','backup_status.json']:
    path=PERSIST/filename
    print(filename,path.read_text() if path.exists() else 'pending')
root=PROJECT/ACTIVE_ROOT
for stage in ['base_zero','base_few','sft','sft_eval','frontier']:
    state=root/stage/'status.json'
    predictions=root/stage/'predictions.jsonl'
    count=sum(1 for line in predictions.open() if line.strip()) if predictions.exists() else 0
    print(stage,'predictions',count,'status',state.read_text() if state.exists() else 'pending')
    log=root/(stage+'.log')
    if log.exists(): print(log.read_text(errors='replace')[-600:])
print('Complete trainer checkpoints:',[p.name for p in (root/'sft/checkpoints').glob('checkpoint-*') if (p/'trainer_state.json').is_file()])


In [ ]:
# Optional separate hosted stage. Never paste a key in notebook code or chat.
if RUN_GEMINI:
    assert BILLING_DISABLED_CONFIRMED,'Verify a Free tier project with billing disabled first'
    from google.colab import userdata
    os.environ['GEMINI_API_KEY']=userdata.get('GEMINI_API_KEY')
    os.environ['SQLRL_GEMINI_FREE_PROJECT']='1'
    try:
        subprocess.run([PY,'scripts/run_free_batched.py','--stage','gemini','--root',ACTIVE_ROOT,'--max-requests',str(MAX_REQUESTS)],cwd=PROJECT,check=True)
    finally:
        os.environ.pop('GEMINI_API_KEY',None)


In [ ]:
# Comparison is produced only after every full-cohort evaluation exists.
needed=[PROJECT/ACTIVE_ROOT/(name+'_evaluation/summary.json') for name in ['base_zero','base_few','sft_eval','frontier']]
if all(path.exists() for path in needed):
    subprocess.run([PY,'scripts/run_free_batched.py','--stage','report','--root',ACTIVE_ROOT],cwd=PROJECT,check=True)
    print((PROJECT/ACTIVE_ROOT/'comparison.md').read_text())
else:
    print('Phases 0–3 are still incomplete. Preserve the verified private backup and resume unfinished stages.')
